# Clean stages 1, 2, and 4: tripartite cycle separators and perturbation search

## Purpose

This notebook carries out the corrected initial construction and the local, time-limited perturbation search discussed for the project. The object of interest is the **full tripartite cycle polytope**

$$
\mathcal{C}_{\mathrm{cycle}}
=
\operatorname{conv}\!\left(
\mathcal{V}_{\circlearrowright}
\cup
\mathcal{V}_{\circlearrowleft}
\right),
$$

where the two vertex sets correspond to the clockwise and counterclockwise communication patterns. The essential correction relative to the earlier implementation is that these two orientations are generated by genuinely different dependency rules; the counterclockwise list is not a duplicate of the clockwise one.

For each of the four nonsignalling extremal representatives in classes 27, 29, 43, and 44, the first part constructs a linear Bell functional $c$ satisfying

$$
c\cdot p \geq 1
\qquad\text{for every }p\in\mathcal{C}_{\mathrm{cycle}},
$$

while the selected nonsignalling point $q$ has $c\cdot q<1$.

The final part performs the perturbation search. It mixes each target with two deterministic vertices of the **cycle-classical polytope**, reruns the same separation LP, removes repeated inequalities, checks validity exactly, certifies the affine dimension of the saturated face, and saves checkpoints and readable candidate tables.

## What is done, in order

1. Fix one explicit coordinate convention for the 64 probabilities $p(abc\mid xyz)$.
2. Generate and audit the deterministic vertices of both cycle orientations.
3. Reconstruct the four published nonsignalling representatives from their correlators.
4. Verify exactly that each target is a normalized, nonsignalling extremal point.
5. Solve one unperturbed separation LP per target.
6. Run the time-limited ordered-pair perturbation search for all four targets.

This notebook still contains **no NPA calculation or see-saw optimization**. Those later stages should consume the audited inequalities saved by this search.

## Requirements and execution

The notebook uses Julia's standard libraries `LinearAlgebra`, `Printf`, `Dates`, and `Serialization`, together with the packages `JuMP` and `GLPK`. Execute the notebook from top to bottom. The final cell is configured for one hour per class, so a complete uninterrupted run takes approximately four hours in addition to setup time.


## 1. Canonical probability coordinates

A behavior in the binary-input, binary-output tripartite scenario consists of

$$
p(abc\mid xyz),
\qquad a,b,c,x,y,z\in\{0,1\}.
$$

It is stored as a vector of length $2^6=64$. To remove any ambiguity, the vector is split into eight input blocks, ordered lexicographically as

$$
xyz=000,001,010,011,100,101,110,111.
$$

Inside each input block, the eight outputs use the same order,

$$
abc=000,001,010,011,100,101,110,111.
$$

Thus $z$ is the fastest-changing input bit and $c$ is the fastest-changing output bit. In Julia's one-based indexing, the coordinate of $p(abc\mid xyz)$ is

$$
1+c+2b+4a+8z+16y+32x.
$$

The binary outputs are also interpreted as signs when correlators are used later: output `0` corresponds to $+1$, and output `1` corresponds to $-1$.

The two assertions at the end of the next cell are small but important regression tests. They verify both the within-block output order and the order of the eight input blocks. Keeping this map explicit prevents the kind of silent iterator-order error that can change a polytope while leaving array dimensions apparently correct.


In [3]:
using LinearAlgebra
using Printf
using JuMP
using GLPK

const CLASSES = (27, 29, 43, 44)
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

#
# The 64-vector is ordered in blocks of fixed (x,y,z):
#
#   (x,y,z) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Within each block, the output order is
#
#   (a,b,c) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Thus c and z are the fastest-changing indices. Binary output 0 corresponds
# to the physicists' value +1, and binary output 1 corresponds to -1.

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)




## 2. Correct deterministic vertices of the two cycle orientations

Each party uses a deterministic binary response function of two binary inputs. There are

$$
2^{2^2}=16
$$

such functions. The integer masks `0:15` encode their four-entry truth tables; `response(mask,u,v)` reads the output assigned to the pair $(u,v)$.

The two communication orientations have different causal dependencies:

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Choosing one of 16 response functions independently for each party gives $16^3=4096$ deterministic behaviors per orientation. Every deterministic behavior has exactly one probability equal to 1 in each input block and all other entries equal to 0.

The polytope used by the LP is the convex hull of the **union** of the two orientation lists. Their intersection contains the 64 fully local deterministic behaviors, so removing duplicate rows gives

$$
4096+4096-64=8128
$$

distinct vertices. Removing duplicates does not change the convex hull or any LP optimum, but it exposes the correct combinatorics and avoids redundant constraints.

This cell performs structural checks before any inequality is computed: binary entries, one selected output per input, expected vertex counts, intersection size, and affine dimensions. The individual orientation polytopes have affine dimension 44; their corrected union has affine dimension 50. In particular, the jump from 44 to 50 is a strong diagnostic that the second orientation was not accidentally generated as a copy of the first.


In [5]:

"""
    response(mask, u, v)

Evaluate one of the 16 deterministic binary functions of two binary inputs.
Bit `2u + v` of `mask` stores the output for the input pair `(u,v)`.
"""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""
    deterministic_vertex(fA, fB, fC, orientation)

Construct one deterministic 64-component behavior in canonical coordinates.

For `:clockwise`:
    a = fA(x,z),  b = fB(y,x),  c = fC(z,y).

For `:counterclockwise`:
    a = fA(x,y),  b = fB(y,z),  c = fC(z,x).
"""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one cycle orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n = size(points, 1)
    n == 0 && return -1
    n == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)

        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# `unique(...; dims=1)` removes the 64 local deterministic behaviors that occur
# in both orientations. Keeping duplicates would not change the LP, but it would
# obscure the actual vertex count and the geometry.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128

@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))




Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Published representatives of NS classes 27, 29, 43, and 44

The four targets are entered through the 26 correlators of a binary tripartite nonsignalling behavior. Their stored order is

$$
\begin{aligned}
&(A_0,A_1;\ B_0,B_1;\ C_0,C_1;\\
&AB_{00},AB_{01},AB_{10},AB_{11};\\
&AC_{00},AC_{01},AC_{10},AC_{11};\\
&BC_{00},BC_{01},BC_{10},BC_{11};\\
&ABC_{000},ABC_{001},ABC_{010},ABC_{011},
ABC_{100},ABC_{101},ABC_{110},ABC_{111}).
\end{aligned}
$$

These representatives are those reported in Table 1 of S. Pironio, J.-D. Bancal, and V. Scarani, *Extremal correlations of the tripartite no-signaling polytope*, J. Phys. A **44**, 065303 (2011), arXiv:1101.2477.

To convert correlators into probabilities, define $s_a=(-1)^a$, and analogously for $b$ and $c$. Then

$$
\begin{aligned}
p(abc\mid xyz)=\frac18[&1+s_aA_x+s_bB_y+s_cC_z
+s_as_bAB_{xy}+s_as_cAC_{xz}\\
&+s_bs_cBC_{yz}+s_as_bs_cABC_{xyz}].
\end{aligned}
$$

The implementation uses Julia rational numbers such as `1//3` throughout this conversion. Consequently, positivity, normalization, nonsignalling, and the later evaluations of rationalized inequalities can be checked exactly rather than up to floating-point tolerance.


In [7]:
#
# Correlator order:
#
#   A0,A1, B0,B1, C0,C1,
#   AB00,AB01,AB10,AB11,
#   AC00,AC01,AC10,AC11,
#   BC00,BC01,BC10,BC11,
#   ABC000,ABC001,ABC010,ABC011,ABC100,ABC101,ABC110,ABC111.
#
# These representatives agree with Table 1 of:
# S. Pironio, J.-D. Bancal, and V. Scarani,
# "Extremal correlations of the tripartite no-signaling polytope",
# J. Phys. A 44, 065303 (2011), arXiv:1101.2477.

const NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 NS correlators to the canonical 64 probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

const NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)




Dict{Int64, Vector{Rational{Int64}}} with 4 entries:
  29 => [2//3, 0, 0, 0, 0, 0, 0, 1//3, 1//3, 1//3  …  0, 0, 0, 1//3, 1//3, 0, 1…
  43 => [2//7, 1//7, 1//7, 0, 0, 1//7, 1//7, 1//7, 2//7, 1//7  …  0, 1//7, 0, 2…
  27 => [1//4, 1//4, 1//4, 0, 0, 0, 0, 1//4, 1//4, 1//4  …  1//4, 0, 0, 1//4, 1…
  44 => [1//4, 0, 0, 1//4, 0, 1//4, 1//4, 0, 1//4, 0  …  1//4, 0, 0, 1//4, 1//4…

## 4. Exact target checks: positivity, normalization, nonsignalling, and extremality

Before using a target in an LP, this cell verifies the defining probabilistic and nonsignalling conditions directly in the 64-coordinate representation:

- every probability is nonnegative;
- the eight probabilities for each fixed input triple $(x,y,z)$ sum to 1;
- the $AB$ marginal is independent of Charlie's input $z$;
- the $AC$ marginal is independent of Bob's input $y$;
- the $BC$ marginal is independent of Alice's input $x$.

It then independently checks that each representative is a **vertex** of the nonsignalling polytope. The normalization and nonsignalling equalities are assembled as a (redundant) $56\times64$ matrix. Its rank is 38, hence the nonsignalling affine space has dimension

$$
64-38=26.
$$

At a given target, every zero probability saturates one positivity inequality. The target is extremal when the equality rows together with these active positivity rows have rank 64: there is then no nonzero feasible direction that keeps all active constraints saturated. The printed `active rank = 64 (NS vertex)` is therefore a geometric check, not merely a lookup of the class label.

All these tests use exact target probabilities. A failed assertion indicates an inconsistent representative, correlator order, probability order, or nonsignalling implementation and should be investigated before proceeding.


In [9]:

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    # Normalization for every input triple.
    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

"""
Return a (redundant) matrix for normalization and tripartite NS equalities.
Its rank must be 38, leaving the familiar 26-dimensional NS affine space.
"""
function nonsignalling_equality_matrix()
    equalities = zeros(Float64, 56, 64)
    row = 0

    for (x, y, z) in TRIPLES
        row += 1
        for (a, b, c) in TRIPLES
            equalities[row, prob_index(a, b, c, x, y, z)] = 1
        end
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        row += 1
        for c in 0:1
            equalities[row, prob_index(a, b, c, x, y, 0)] += 1
            equalities[row, prob_index(a, b, c, x, y, 1)] -= 1
        end
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        row += 1
        for b in 0:1
            equalities[row, prob_index(a, b, c, x, 0, z)] += 1
            equalities[row, prob_index(a, b, c, x, 1, z)] -= 1
        end
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        row += 1
        for a in 0:1
            equalities[row, prob_index(a, b, c, 0, y, z)] += 1
            equalities[row, prob_index(a, b, c, 1, y, z)] -= 1
        end
    end

    @assert row == 56
    return equalities
end

const NS_EQUALITIES = nonsignalling_equality_matrix()
@assert rank(NS_EQUALITIES; atol=1e-9, rtol=0.0) == 38

"""
Check extremality in the NS polytope using the rank of the NS equalities plus
the positivity inequalities saturated by the target.
"""
function ns_active_rank(probabilities)
    zero_coordinates = findall(iszero, probabilities)
    coordinate_rows = Matrix{Float64}(I, 64, 64)[zero_coordinates, :]
    active_matrix = vcat(NS_EQUALITIES, coordinate_rows)
    return rank(active_matrix; atol=1e-9, rtol=0.0)
end

println("\nTarget checks")
for class in CLASSES
    target = NS_TARGETS[class]
    validate_nonsignalling_behavior(target)
    active_rank = ns_active_rank(target)
    @assert active_rank == 64

    @printf(
        "  class %2d: support = %2d, active rank = %2d (NS vertex)\n",
        class,
        count(value -> !iszero(value), target),
        active_rank,
    )
end





Target checks
  class 27: support = 32, active rank = 64 (NS vertex)
  class 29: support = 23, active rank = 64 (NS vertex)
  class 43: support = 36, active rank = 64 (NS vertex)
  class 44: support = 32, active rank = 64 (NS vertex)


## 5. Separation LP for the corrected full cycle polytope

For each nonsignalling target $q$, the code searches for a nonnegative coefficient vector $c\in\mathbb{R}^{64}$ by solving

$$
\begin{aligned}
\text{minimize}\quad & c\cdot q\\
\text{subject to}\quad & c\cdot v\geq 1
&&\text{for all }v\in\mathcal{V}_{\mathrm{cycle}},\\
& c_j\geq0
&&\text{for }j=1,\ldots,64.
\end{aligned}
$$

The bound is normalized to 1. Since a linear functional attains its minimum over a polytope at a vertex, imposing the first constraint on all 8128 deterministic vertices proves

$$
c\cdot p\geq1
\qquad\forall p\in\mathcal{C}_{\mathrm{cycle}}.
$$

Therefore an optimal target value below 1 is a certificate that $q$ lies outside the cycle polytope. The LP is constrained by the corrected **full union**, not by one orientation alone. After solving, the code also reports the minima on the clockwise and counterclockwise subsets separately; both must respect the bound.

GLPK returns floating-point coefficients. The code reconstructs simple rational coefficients, reevaluates every vertex exactly, and requires the exact classical minimum to be 1. It then studies the vertices saturating $c\cdot v=1$. Because the full cycle polytope has affine dimension 50, a supporting inequality is facet-defining precisely when the saturated face has affine dimension 49.

Finally, the exact target values

| class | $c\cdot q$ |
|---:|---:|
| 27 | $3/4$ |
| 29 | $1/2$ |
| 43 | $5/7$ |
| 44 | $0$ |

are asserted as independent regression values. They do **not** replace the validity checks on both orientations: a scalar optimum can remain unchanged even when an LP was built from the wrong vertex set, whereas the associated optimizing coefficient vector need not remain valid.


In [11]:
#
# For each target q, solve
#
#   minimize    c . q
#   subject to  c . v >= 1    for every one of the 8128 cycle vertices,
#               c >= 0.
#
# A value below 1 gives the valid separating inequality c . p >= 1.

function rationalize_coefficients(coefficients; tol=1e-8)
    return [
        abs(value) <= tol ? 0//1 : rationalize(Int, value; tol=tol)
        for value in coefficients
    ]
end

function solve_separator(
    class::Int,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    tol=1e-8,
    verbose=false,
)
    n_vertices, dimension = size(vertices_cycle)
    @assert dimension == length(target) == 64

    model = Model(GLPK.Optimizer)
    verbose || set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @objective(
        model,
        Min,
        sum(Float64(target[j]) * coefficients[j] for j in 1:dimension),
    )
    @constraint(
        model,
        classical[i=1:n_vertices],
        sum(vertices_cycle[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )

    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("LP for class $class ended with status $status")

    coefficients_float = value.(coefficients)
    coefficients_exact = rationalize_coefficients(coefficients_float; tol=tol)

    values_cycle = Float64.(vertices_cycle) * coefficients_float
    values_clockwise = Float64.(vertices_clockwise) * coefficients_float
    values_counterclockwise = Float64.(vertices_counterclockwise) * coefficients_float

    exact_cycle_values = [
        sum(
            coefficients_exact[j] * Int(vertices_cycle[i, j])
            for j in 1:dimension
        )
        for i in 1:n_vertices
    ]
    exact_target_value = sum(
        coefficients_exact[j] * target[j]
        for j in 1:dimension
    )

    exact_classical_minimum = minimum(exact_cycle_values)
    exact_classical_minimum == 1 || error(
        "Rational reconstruction failed for class $class: exact minimum is " *
        "$exact_classical_minimum instead of 1.",
    )

    saturating_indices = findall(==(1//1), exact_cycle_values)
    face_dimension = affine_dimension(vertices_cycle[saturating_indices, :])

    return (
        class=class,
        status=status,
        coefficients=coefficients_float,
        exact_coefficients=coefficients_exact,
        solver_target_value=objective_value(model),
        exact_target_value=exact_target_value,
        classical_minimum=minimum(values_cycle),
        clockwise_minimum=minimum(values_clockwise),
        counterclockwise_minimum=minimum(values_counterclockwise),
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=(face_dimension == affine_dimension(vertices_cycle) - 1),
    )
end

separator_results = Dict{Int, Any}()

for class in CLASSES
    separator_results[class] = solve_separator(
        class,
        NS_TARGETS[class],
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )
end

# Independent expected values. These assertions guard against future changes in
# coordinate order, target conversion, or vertex generation.
const EXPECTED_TARGET_VALUES = Dict(
    27 => 3//4,
    29 => 1//2,
    43 => 5//7,
    44 => 0//1,
)

for class in CLASSES
    @assert separator_results[class].exact_target_value == EXPECTED_TARGET_VALUES[class]
end

println("\nCorrected separation results")
println("class  target   min(CW)   min(CCW)  saturating  face-dim  facet")
for class in CLASSES
    result = separator_results[class]
    @printf(
        "%5d  %7s   %7.4f   %8.4f  %10d  %8d  %s\n",
        class,
        string(result.exact_target_value),
        result.clockwise_minimum,
        result.counterclockwise_minimum,
        result.saturation_count,
        result.face_dimension,
        string(result.is_facet),
    )
end





Corrected separation results
class  target   min(CW)   min(CCW)  saturating  face-dim  facet
   27     3//4    1.0000     1.0000         119        49  true
   29     1//2    1.0000     1.0000          90        44  false
   43     5//7    1.0000     1.0000         508        49  true
   44     0//1    1.0000     1.0000         508        49  true


## 6. Inequality coefficient matrices

The final cell reshapes each exact 64-component coefficient vector into an $8\times8$ matrix for reading and reuse. Both axes follow the binary lexicographic order

$$
000,001,010,011,100,101,110,111.
$$

- Row `xyz` fixes the input triple.
- Column `abc` fixes the output triple.
- The entry in that row and column is the coefficient multiplying $p(abc\mid xyz)$.

Thus, if the displayed matrix for a class is denoted by $C$, its inequality is

$$
\sum_{x,y,z,a,b,c}
C_{xyz,abc}\,p(abc\mid xyz)\geq1.
$$

The matrices contain exact rational numbers and are generated from the coefficients that have already passed the full-vertex validity check. They are the appropriate first-stage outputs to carry into later quantum-bound and see-saw analyses.


In [13]:
# Rows and columns both follow 000,001,010,011,100,101,110,111.
# Rows label inputs (x,y,z), columns label outputs (a,b,c).

coefficient_matrix(coefficients) =
    Matrix(permutedims(reshape(coefficients, 8, 8)))

inequality_matrices = Dict(
    class => coefficient_matrix(separator_results[class].exact_coefficients)
    for class in CLASSES
)

println("\nExact coefficient matrices")
for class in CLASSES
    println("\nClass $class:  c . p >= 1")
    display(inequality_matrices[class])
end



Exact coefficient matrices

Class 27:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 1//2  1//2  1//2   1    1//2  1  1  0
  0     0     0    1//2  1//2  1  1  0
  0     0     0    1//2  1//2  1  1  0
 1//2  1//2  1//2   1    1//2  1  1  0
  0     1     1     0    1//2  0  0  1
  0     1     1     0    1//2  0  0  1
  0     1     1     0    1//2  0  0  1
 1//2   0     0     1     0    1  1  0


Class 29:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
  0    1//2  1//2   1    1//2   1     1    0
  0    1//2   1     1     1     1     0    1
  0     1    1//2   1     1     0     1    1
 1//2   0     0     0     0     1     1    0
  0     1     1     0    1//2   1     1    1
 1//2   0     0     1     0     0     1    0
 1//2   0     0     1     0     1     0    0
  1     0     0    1//2   0    1//2  1//2  1


Class 43:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0


Class 44:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 1  0  0  1  0  1  1  0

# Stage 4: time-limited perturbation search

## Mathematical procedure

For a nonsignalling target $q$ and two **distinct, ordered** deterministic cycle vertices $v_j$ and $v_k$, define

$$
q_{jk}^{(\epsilon)}
=
\left(1-\frac{3\epsilon}{2}\right)q
+\epsilon v_j
+\frac{\epsilon}{2}v_k.
$$

The previous implementation used $\epsilon=10^{-3}$, which is retained here. The weights add to one, and the unequal coefficients of $v_j$ and $v_k$ mean that $(j,k)$ and $(k,j)$ are different perturbations.

The points $v_j,v_k$ range over all 8128 deterministic vertices of the corrected cycle-classical polytope. They are not restricted to the 64 ordinary Bell-local vertices. General cycle vertices may be signalling in the ordinary Bell sense; the perturbed point is only an auxiliary objective direction and is not claimed to be a physical quantum behavior.

For each perturbed target, the program solves

$$
\begin{aligned}
\text{minimize}\quad &c\cdot q_{jk}^{(\epsilon)}\\
\text{subject to}\quad &c\cdot v_i\geq1
&&\text{for all }i=1,\ldots,8128,\\
&c_\ell\geq0
&&\text{for }\ell=1,\ldots,64.
\end{aligned}
$$

Only the objective changes from one pair to the next. The 8128 classical constraints remain fixed, so the JuMP model is built once and reused.

## What counts as a saved candidate

The program does **not** save one record for every ordered pair. Many pairs return the same inequality. Instead, it saves every distinct inequality after:

1. reconstructing rational coefficients;
2. renormalizing the exact classical minimum to 1;
3. checking the inequality exactly on all corrected cycle vertices; and
4. removing duplicate normalized coefficient vectors.

Every retained inequality receives an exact affine face dimension and an `is_facet` label. Facets have face dimension 49 because the corrected full cycle polytope has affine dimension 50. Nonfacets are retained rather than discarded, so the later quantum analysis may decide whether to inspect them.


In [15]:
using Dates
using Serialization

# The four large primes below are used to compute affine ranks exactly.
# They are pairwise distinct primes and fit safely in Int64. Products are
# evaluated through Int128 so that modular multiplication cannot overflow.
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent

    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end

    return result
end

"""Compute the rank of an integer matrix exactly over the field Z/modulus*Z."""
function rank_mod_prime(matrix::AbstractMatrix{<:Integer}, modulus::Int64)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)

    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1

    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(
            work[pivot_row, column],
            modulus - 2,
            modulus,
        )

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue

            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(factor, work[pivot_row, j], modulus)
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end

        pivot_row += 1
    end

    return pivot_row - 1
end

"""
    certified_integer_rank(matrix)

Return the exact rational rank of an integer matrix.

For each prime, modular rank is a lower bound on rational rank. If the largest
modular rank is r, every (r+1)-minor is divisible by the product of all primes.
The Hadamard bound computed below is smaller than that product, proving that
all those minors vanish over the integers. Thus r is also an upper bound.
"""
function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)

    # Full row or column rank is already exact once witnessed modulo a prime.
    certified_rank == min(n_rows, n_columns) && return certified_rank

    # Every row in the matrices used here contains integers. Hadamard's bound
    # says that the square of any k-by-k determinant is no larger than the
    # product of the squared Euclidean norms of its rows.
    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )

    return certified_rank
end

"""Compute the exact affine dimension of a set of integer points."""
function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points == 0 && return -1
    n_points == 1 && return 0

    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end

    return certified_integer_rank(differences)
end

# This is evaluated only once. Besides protecting the facet test, it gives an
# exact independent confirmation of the dimension found numerically above.
@assert certified_affine_dimension(vertices_cycle) == 50

"""Construct the exact perturbed objective vector for one ordered pair."""
function perturb_target(target, vertex_j, vertex_k, epsilon::Rational{Int})
    target_weight = 1 - 3 * epsilon / 2
    target_weight >= 0 || error("epsilon must satisfy 1 - 3epsilon/2 >= 0")

    return (
        target_weight .* target .+
        epsilon .* Int.(vertex_j) .+
        (epsilon / 2) .* Int.(vertex_k)
    )
end

"""Build the fixed separation LP once; only its objective will be updated."""
function build_perturbation_model(vertices)
    n_vertices, dimension = size(vertices)
    @assert dimension == 64

    model = Model(GLPK.Optimizer)
    set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @constraint(
        model,
        classical[i=1:n_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )

    # A linear objective must exist before its coefficients can be modified.
    @objective(model, Min, sum(coefficients))
    return model, coefficients
end

"""Solve the fixed LP after replacing all 64 objective coefficients."""
function solve_perturbed_target!(model, coefficients, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    optimize!(model)

    status = termination_status(model)
    status == MOI.OPTIMAL || return nothing, status, NaN
    return value.(coefficients), status, objective_value(model)
end

"""Evaluate a rational coefficient vector exactly on every supplied vertex."""
function exact_vertex_values(vertices, coefficients)
    n_vertices, dimension = size(vertices)
    @assert dimension == length(coefficients)

    return Rational{BigInt}[
        sum(
            coefficients[column] * Int(vertices[row, column])
            for column in 1:dimension
        )
        for row in 1:n_vertices
    ]
end

"""
Convert one floating-point LP output into an exact, normalized, audited record.

The rationalized inequality is divided by its exact minimum on the full cycle
vertex set. Consequently, every returned candidate has exact classical bound 1.
"""
function audit_candidate(
    class::Int,
    pair_j::Int,
    pair_k::Int,
    coefficients_float,
    original_target,
    perturbed_target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    rational_tolerance=1e-8,
)
    all(isfinite, coefficients_float) || return nothing
    any(value -> value < -rational_tolerance, coefficients_float) && return nothing

    coefficients_rational = Rational{BigInt}[
        abs(value) <= rational_tolerance ?
            BigInt(0)//BigInt(1) :
            rationalize(BigInt, max(value, 0.0); tol=rational_tolerance)
        for value in coefficients_float
    ]

    cycle_values = exact_vertex_values(vertices_cycle, coefficients_rational)
    raw_minimum = minimum(cycle_values)
    raw_minimum > 0 || return nothing

    exact_coefficients = coefficients_rational ./ raw_minimum
    cycle_values ./= raw_minimum
    minimum(cycle_values) == 1 || error("Exact normalization failed")

    saturating_indices = findall(==(1//1), cycle_values)
    face_dimension = certified_affine_dimension(
        vertices_cycle[saturating_indices, :],
    )

    clockwise_minimum = minimum(exact_vertex_values(
        vertices_clockwise,
        exact_coefficients,
    ))
    counterclockwise_minimum = minimum(exact_vertex_values(
        vertices_counterclockwise,
        exact_coefficients,
    ))

    original_value = sum(exact_coefficients .* original_target)
    perturbed_value = sum(exact_coefficients .* perturbed_target)

    return (
        class=class,
        first_j=pair_j,
        first_k=pair_k,
        exact_coefficients=exact_coefficients,
        original_target_value=original_value,
        perturbed_target_value=perturbed_value,
        separates_original=(original_value < 1),
        separates_perturbed=(perturbed_value < 1),
        classical_minimum=1//1,
        clockwise_minimum=clockwise_minimum,
        counterclockwise_minimum=counterclockwise_minimum,
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=(face_dimension == 49),
    )
end


audit_candidate

## Ordered search, deduplication, and checkpoints

The next cell implements the same deterministic order as the old code:

```text
for j in 1:8128
    for k in 1:8128
        j == k && continue
        ...
    end
end
```

The complete corrected scan contains 66,056,256 LPs per NS class. The local run therefore stops after a cumulative wall-clock budget rather than attempting exhaustion.

For each class, the program creates two files inside `cycle_perturbation_results`:

- `class_XX_checkpoint.jls`: the complete machine-readable state needed to resume;
- `class_XX_candidates.tsv`: a human-readable table containing every distinct audited inequality found so far.

The checkpoint is written atomically every five minutes and once more when the function finishes or is interrupted. Reexecuting the final cell resumes at the stored pair and uses only the unspent part of the one-hour budget. To extend a completed search, increase `TARGET_SECONDS_PER_CLASS`; for example, changing it from `3600.0` to `7200.0` extends every class to two cumulative hours.

Deduplication has two layers. A rounded floating-point key cheaply rejects the many immediate repetitions; the final identity test uses the complete tuple of exact, bound-normalized rational coefficients.


In [17]:
function candidate_paths(output_directory, class::Int)
    prefix = "class_" * lpad(string(class), 2, "0")
    return (
        checkpoint=joinpath(output_directory, prefix * "_checkpoint.jls"),
        table=joinpath(output_directory, prefix * "_candidates.tsv"),
    )
end

function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

"""Write a readable table without requiring an external CSV package."""
function write_candidate_table(path, candidates)
    open(path, "w") do io
        println(
            io,
            join(
                [
                    "class",
                    "first_j",
                    "first_k",
                    "original_target_value",
                    "perturbed_target_value",
                    "separates_original",
                    "separates_perturbed",
                    "saturation_count",
                    "face_dimension",
                    "is_facet",
                    "coefficients_in_canonical_order",
                ],
                "\t",
            ),
        )

        for candidate in candidates
            coefficient_text = join(
                rational_string.(candidate.exact_coefficients),
                ",",
            )
            println(
                io,
                join(
                    string.([
                        candidate.class,
                        candidate.first_j,
                        candidate.first_k,
                        rational_string(candidate.original_target_value),
                        rational_string(candidate.perturbed_target_value),
                        candidate.separates_original,
                        candidate.separates_perturbed,
                        candidate.saturation_count,
                        candidate.face_dimension,
                        candidate.is_facet,
                        coefficient_text,
                    ]),
                    "\t",
                ),
            )
        end
    end
    return path
end

"""Write a checkpoint through a temporary file to avoid partial state files."""
function write_checkpoint(path, state)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        serialize(io, state)
    end
    mv(temporary_path, path; force=true)
    return path
end

function read_checkpoint(path)
    return open(path, "r") do io
        deserialize(io)
    end
end

function next_ordered_pair(j::Int, k::Int, n_vertices::Int)
    next_j = j
    next_k = k + 1
    if next_k > n_vertices
        next_j += 1
        next_k = 1
    end
    return next_j, next_k
end

function rough_coefficient_key(coefficients; digits=8, zero_tolerance=1e-9)
    return Tuple(
        abs(value) <= zero_tolerance ? 0.0 : round(value; digits=digits)
        for value in coefficients
    )
end

"""
    run_class_search!(class, model, coefficients, ...)

Continue the deterministic ordered-pair search for one NS class until its
cumulative time reaches `target_seconds`. Existing checkpoints are resumed.
"""
function run_class_search!(
    class::Int,
    model,
    coefficients,
    original_target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    epsilon=1//1000,
    target_seconds=3600.0,
    checkpoint_interval_seconds=300.0,
    progress_every=500,
    output_directory=joinpath(pwd(), "cycle_perturbation_results"),
    resume=true,
)
    mkpath(output_directory)
    paths = candidate_paths(output_directory, class)
    n_vertices = size(vertices_cycle, 1)

    if resume && isfile(paths.checkpoint)
        state = read_checkpoint(paths.checkpoint)
        state["format_version"] == 1 || error("Unknown checkpoint format")
        state["class"] == class || error("Checkpoint class mismatch")
        state["epsilon"] == epsilon || error("Checkpoint epsilon mismatch")
        state["n_vertices"] == n_vertices || error("Checkpoint vertex-count mismatch")

        pair_j = state["next_j"]
        pair_k = state["next_k"]
        total_lp_solves = state["total_lp_solves"]
        total_failures = state["total_failures"]
        elapsed_before_call = state["elapsed_seconds"]
        candidates = state["candidates"]
        seen_rough = state["seen_rough"]

        println(
            "Resuming class $class at (j,k)=($pair_j,$pair_k) with " *
            "$(length(candidates)) distinct candidates.",
        )
    else
        pair_j = 1
        pair_k = 1
        total_lp_solves = 0
        total_failures = 0
        elapsed_before_call = 0.0
        candidates = Any[]
        seen_rough = Set{Any}()
    end

    seen_exact = Set{Any}(
        Tuple(candidate.exact_coefficients) for candidate in candidates
    )

    remaining_seconds = max(0.0, target_seconds - elapsed_before_call)
    if remaining_seconds == 0.0 || pair_j > n_vertices
        println(
            "Class $class already reached its requested budget or exhausted all pairs.",
        )
        return candidates
    end

    started_at = time()
    last_checkpoint_at = started_at

    function elapsed_total()
        return elapsed_before_call + (time() - started_at)
    end

    function current_state()
        return Dict{String, Any}(
            "format_version" => 1,
            "class" => class,
            "epsilon" => epsilon,
            "n_vertices" => n_vertices,
            "next_j" => pair_j,
            "next_k" => pair_k,
            "total_lp_solves" => total_lp_solves,
            "total_failures" => total_failures,
            "elapsed_seconds" => min(elapsed_total(), target_seconds),
            "candidates" => candidates,
            "seen_rough" => seen_rough,
            "saved_at" => string(now()),
        )
    end

    try
        while pair_j <= n_vertices && elapsed_total() < target_seconds
            if pair_j == pair_k
                pair_j, pair_k = next_ordered_pair(pair_j, pair_k, n_vertices)
                continue
            end

            current_j = pair_j
            current_k = pair_k

            try
                target_perturbed = perturb_target(
                    original_target,
                    view(vertices_cycle, current_j, :),
                    view(vertices_cycle, current_k, :),
                    epsilon,
                )

                coefficients_float, status, solver_objective =
                    solve_perturbed_target!(
                        model,
                        coefficients,
                        target_perturbed,
                    )
                total_lp_solves += 1

                if coefficients_float === nothing
                    total_failures += 1
                    @warn "LP did not terminate optimally" class current_j current_k status
                else
                    rough_key = rough_coefficient_key(coefficients_float)

                    if !(rough_key in seen_rough)
                        push!(seen_rough, rough_key)

                        candidate = audit_candidate(
                            class,
                            current_j,
                            current_k,
                            coefficients_float,
                            original_target,
                            target_perturbed,
                            vertices_cycle,
                            vertices_clockwise,
                            vertices_counterclockwise,
                        )

                        if candidate !== nothing
                            exact_key = Tuple(candidate.exact_coefficients)
                            if !(exact_key in seen_exact)
                                push!(seen_exact, exact_key)
                                push!(candidates, candidate)

                                label = candidate.is_facet ? "FACET" : "nonfacet"
                                println(
                                    "Class $class: new $label from " *
                                    "(j,k)=($current_j,$current_k), " *
                                    "face dimension=$(candidate.face_dimension), " *
                                    "q-value=$(candidate.original_target_value), " *
                                    "perturbed LP objective=$solver_objective",
                                )
                            end
                        end
                    end
                end
            catch error_value
                error_value isa InterruptException && rethrow()
                total_failures += 1
                @warn "Error while processing ordered pair" class current_j current_k exception=(error_value, catch_backtrace())
            end

            pair_j, pair_k = next_ordered_pair(pair_j, pair_k, n_vertices)

            if total_lp_solves > 0 && total_lp_solves % progress_every == 0
                facet_count = count(candidate -> candidate.is_facet, candidates)
                @printf(
                    "Class %d | next=(%d,%d) | LPs=%d | unique=%d | facets=%d | elapsed=%.1f min\n",
                    class,
                    pair_j,
                    pair_k,
                    total_lp_solves,
                    length(candidates),
                    facet_count,
                    elapsed_total() / 60,
                )
            end

            if time() - last_checkpoint_at >= checkpoint_interval_seconds
                write_checkpoint(paths.checkpoint, current_state())
                write_candidate_table(paths.table, candidates)
                last_checkpoint_at = time()
            end
        end
    finally
        write_checkpoint(paths.checkpoint, current_state())
        write_candidate_table(paths.table, candidates)
    end

    facet_count = count(candidate -> candidate.is_facet, candidates)
    println(
        "Finished current budget for class $class: " *
        "$(length(candidates)) distinct candidates, $facet_count facets, " *
        "$total_lp_solves total LP solves.",
    )
    println("Results directory: ", abspath(output_directory))

    return candidates
end


run_class_search!

## Run configuration

The final cell starts the actual search. Its defaults reproduce the agreed local workflow:

- classes 27, 29, 43, and 44, in that order;
- the old nested order of pairs, with `j` outside and `k` inside;
- $\epsilon=10^{-3}$;
- one cumulative hour per class;
- checkpoints every five minutes;
- all distinct valid inequalities retained, with facets identified by exact face dimension 49.

Running the complete cell can take approximately four hours. A progress line is printed every 500 successfully submitted LPs. You may interrupt the kernel safely; the `finally` block attempts one last checkpoint, and rerunning this cell resumes from it.

For a short installation test, temporarily set `TARGET_SECONDS_PER_CLASS = 60.0`. After that test, restore `3600.0`: because the time is cumulative, each class will then run only the remaining 59 minutes.


In [19]:
# -------------------------- USER CONFIGURATION --------------------------

CLASSES_TO_SEARCH = collect(CLASSES)
PERTURBATION_EPSILON = 1//1000
TARGET_SECONDS_PER_CLASS = 120.0
CHECKPOINT_INTERVAL_SECONDS = 300.0
PROGRESS_EVERY = 500
SEARCH_OUTPUT_DIRECTORY = joinpath(pwd(), "cycle_perturbation_results")

# ------------------------------------------------------------------------

search_model, search_coefficients = build_perturbation_model(vertices_cycle)
perturbation_results = Dict{Int, Any}()

println("Perturbation results will be written to:")
println(abspath(SEARCH_OUTPUT_DIRECTORY))

for class in CLASSES_TO_SEARCH
    println("\n", repeat("=", 72))
    println("Starting or resuming perturbation search for class $class")
    println(repeat("=", 72))

    perturbation_results[class] = run_class_search!(
        class,
        search_model,
        search_coefficients,
        NS_TARGETS[class],
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise;
        epsilon=PERTURBATION_EPSILON,
        target_seconds=TARGET_SECONDS_PER_CLASS,
        checkpoint_interval_seconds=CHECKPOINT_INTERVAL_SECONDS,
        progress_every=PROGRESS_EVERY,
        output_directory=SEARCH_OUTPUT_DIRECTORY,
        resume=true,
    )
end

println("\nSearch summary")
println("class  distinct candidates  facets  separating original target")
for class in CLASSES_TO_SEARCH
    candidates = perturbation_results[class]
    @printf(
        "%5d  %19d  %6d  %26d\n",
        class,
        length(candidates),
        count(candidate -> candidate.is_facet, candidates),
        count(candidate -> candidate.separates_original, candidates),
    )
end


Perturbation results will be written to:
C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_perturbation_results

Starting or resuming perturbation search for class 27
Resuming class 27 at (j,k)=(198,5724) with 6 distinct candidates.
Class 27 | next=(198,5983) | LPs=1607000 | unique=6 | facets=6 | elapsed=120.0 min
Class 27 | next=(198,6483) | LPs=1607500 | unique=6 | facets=6 | elapsed=120.1 min
Class 27 | next=(198,6983) | LPs=1608000 | unique=6 | facets=6 | elapsed=120.1 min
Class 27 | next=(198,7483) | LPs=1608500 | unique=6 | facets=6 | elapsed=120.2 min
Class 27 | next=(198,7983) | LPs=1609000 | unique=6 | facets=6 | elapsed=120.2 min
Class 27 | next=(199,356) | LPs=1609500 | unique=6 | facets=6 | elapsed=120.3 min
Class 27 | next=(199,856) | LPs=1610000 | unique=6 | facets=6 | elapsed=120.3 min
Class 27 | next=(199,1356) | LPs=1610500 | unique=6 | facets=6 | elapsed=1

Excessive output truncated after 524351 bytes.

In [20]:
using InteractiveUtils
using Pkg

versioninfo()
Pkg.status()

println("\nCheckpoint diagnostics")
for class in CLASSES
    paths = candidate_paths(SEARCH_OUTPUT_DIRECTORY, class)

    state = open(paths.checkpoint, "r") do io
        deserialize(io)
    end

    println(
        "class = ", class,
        ", LP solves = ", state["total_lp_solves"],
        ", failures = ", state["total_failures"],
        ", elapsed seconds = ", state["elapsed_seconds"],
        ", next pair = (", state["next_j"], ", ", state["next_k"], ")",
        ", candidates = ", length(state["candidates"]),
    )
end

Julia Version 1.12.6
Commit 15346901f0 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Windows (x86_64-w64-mingw32)
  CPU: 12 × 12th Gen Intel(R) Core(TM) i5-12450HX
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, alderlake)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 12 virtual cores)
Status `C:\Users\josen\.julia\environments\v1.12\Project.toml`
  [60bf3e95] GLPK v1.2.1
  [7073ff75] IJulia v1.34.4
⌃ [4076af6c] JuMP v1.30.1
  [b8f27783] MathOptInterface v1.54.0
  [92933f4c] ProgressMeter v1.11.0
  [5cd12e4b] QuantumNPA v0.1.2
  [c946c3f1] SCS v2.7.0
Info Packages marked with ⌃ have new versions available and may be upgradable.

Checkpoint diagnostics
class = 27, LP solves = 4215942, failures = 0, elapsed seconds = 18000.0, next pair = (519, 6158), candidates = 6
class = 29, LP solves = 4701037, failures = 0, elapsed seconds = 18000.0, next pair = (579, 3633), candidates = 12
class = 43, LP solves = 5538999, fai